# PMLDL-finetuning

Fine-tuning **DeBERTa-v3-base** as a cross-encoder for three-class response preference classification.

The pipeline uses swap augmentation, a 1024-token input, one shared prompt, and balanced head-tail truncation of responses A and B.


## 1. Setup

Configuration for data paths, training, tuning, and output files.

In [1]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 67
set_seed(SEED)

TRAIN_PATH = "/kaggle/input/datasets/shanmay/pmldl-finetuning/train.csv"
TEST_PATH = "/kaggle/input/datasets/shanmay/pmldl-finetuning/test.csv"
MODEL_NAME = "microsoft/deberta-v3-base"

MAX_LENGTH = 1024
PRETRAINED_RELATIVE_WINDOW = 512
MAX_PROMPT_TOKENS = 256
TAIL_FRACTION = 0.25
VALID_SIZE = 0.10

FINAL_EPOCHS = 2
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 16
WEIGHT_DECAY = 0.01
EVAL_STEPS = 100
SAVE_STEPS = 1500
SAVE_TOTAL_LIMIT = 2

RUN_HP_TUNING = False
HP_TRAIN_FRACTION = 0.08
HP_VALID_FRACTION = 0.35
HP_EPOCHS = 1

HP_SEARCH_SPACE = [
    {"learning_rate": 1e-5, "classifier_dropout": 0.10, "label_smoothing_factor": 0.00},
    {"learning_rate": 2e-5, "classifier_dropout": 0.10, "label_smoothing_factor": 0.05},
    {"learning_rate": 2e-5, "classifier_dropout": 0.20, "label_smoothing_factor": 0.05},
]

DEFAULT_HP = {
    "learning_rate": 2e-5,
    "classifier_dropout": 0.10,
    "label_smoothing_factor": 0.05,
}

# Set a checkpoint path here to continue training from a previous Kaggle output.
RESUME_FROM_CHECKPOINT = "/kaggle/working/checkpoint-3000-fixed"

OUTPUT_DIR = Path("/kaggle/working")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
FINAL_MODEL_DIR = OUTPUT_DIR / "deberta-v3-base-cross-encoder-1024-smart"
AUGMENTED_DATA_PATH = OUTPUT_DIR / "train_swap_augmented.parquet"
HP_RESULTS_PATH = OUTPUT_DIR / "hp_tuning_results.csv"
BEST_HP_PATH = OUTPUT_DIR / "best_hyperparameters.json"

USE_FP16 = torch.cuda.is_available()

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())

PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU: Tesla T4
GPU count: 2


In [2]:
from pathlib import Path
import shutil

from safetensors import safe_open
from safetensors.torch import load_file, save_file

SRC = Path(
    "/kaggle/input/datasets/shanmay/deberta-checkpoint/checkpoint-3000"
)

DST = Path(
    "/kaggle/working/checkpoint-3000-fixed"
)

# Копируем весь checkpoint:
# optimizer, scheduler, trainer_state, scaler, rng и т.д.
if DST.exists():
    shutil.rmtree(DST)

shutil.copytree(SRC, DST)

src_model = SRC / "model.safetensors"
dst_model = DST / "model.safetensors"

# Сохраняем metadata safetensors
with safe_open(src_model, framework="pt", device="cpu") as f:
    metadata = f.metadata()

state = load_file(src_model, device="cpu")

fixed_state = {}
renamed = 0

for key, value in state.items():
    new_key = key

    if ".LayerNorm.gamma" in new_key:
        new_key = new_key.replace(
            ".LayerNorm.gamma",
            ".LayerNorm.weight"
        )
        renamed += 1

    if ".LayerNorm.beta" in new_key:
        new_key = new_key.replace(
            ".LayerNorm.beta",
            ".LayerNorm.bias"
        )
        renamed += 1

    if new_key in fixed_state:
        raise RuntimeError(f"Duplicate key after rename: {new_key}")

    fixed_state[new_key] = value

save_file(
    fixed_state,
    dst_model,
    metadata=metadata
)

print("Renamed:", renamed)
print("Fixed checkpoint:", DST)

Renamed: 52
Fixed checkpoint: /kaggle/working/checkpoint-3000-fixed


## 2. Load data

Only the columns used for training and inference are loaded.

In [3]:
train_columns = [
    "id", "prompt", "response_a", "response_b",
    "winner_model_a", "winner_model_b", "winner_tie",
]
test_columns = ["id", "prompt", "response_a", "response_b"]

train_df = pd.read_csv(TRAIN_PATH, usecols=train_columns)
test_df = pd.read_csv(TEST_PATH, usecols=test_columns)

print("Train:", train_df.shape)
print("Test :", test_df.shape)

Train: (57477, 7)
Test : (3, 4)


## 3. Labels and validation split

Labels are mapped to `0 = A`, `1 = B`, `2 = tie`.

The validation split is created before augmentation to avoid leakage between original and swapped samples.

In [4]:
LABEL_COLUMNS = ["winner_model_a", "winner_model_b", "winner_tie"]

assert (train_df[LABEL_COLUMNS].sum(axis=1) == 1).all()

train_df["labels"] = np.argmax(
    train_df[LABEL_COLUMNS].to_numpy(),
    axis=1,
).astype(np.int64)

train_df = train_df[["id", "prompt", "response_a", "response_b", "labels"]].copy()

train_part, valid_part = train_test_split(
    train_df,
    test_size=VALID_SIZE,
    random_state=SEED,
    stratify=train_df["labels"],
)

train_part = train_part.reset_index(drop=True)
valid_part = valid_part.reset_index(drop=True)

print("Train split:", train_part.shape)
print("Valid split:", valid_part.shape)

Train split: (51729, 5)
Valid split: (5748, 5)


## 4. Swap augmentation

Each training sample is duplicated with responses A and B exchanged.

Labels are swapped as `A ↔ B`, while `tie` remains unchanged. The doubled dataset is shuffled and saved.

In [5]:
def swap_ab(frame):
    swapped = frame.copy()
    swapped[["response_a", "response_b"]] = frame[["response_b", "response_a"]].to_numpy()
    swapped["labels"] = frame["labels"].map({0: 1, 1: 0, 2: 2}).astype(np.int64)
    return swapped


swapped_train = swap_ab(train_part)
train_aug = pd.concat([train_part, swapped_train], ignore_index=True)
train_aug = train_aug.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

assert len(train_aug) == 2 * len(train_part)

train_aug.to_parquet(AUGMENTED_DATA_PATH, index=False)

print("Augmented train:", train_aug.shape)
print("Saved:", AUGMENTED_DATA_PATH)

Augmented train: (103458, 5)
Saved: /kaggle/working/train_swap_augmented.parquet


## 5. Cross-encoder input

The prompt is included only once. The model receives one sequence containing `prompt`, `response A`, and `response B`.

Long examples are truncated explicitly during tokenization so that neither response can consume the whole context window.


In [6]:
def parse_turns(value):
    if isinstance(value, list):
        turns = value
    elif pd.isna(value):
        turns = []
    else:
        text = str(value)
        try:
            parsed = json.loads(text)
            turns = parsed if isinstance(parsed, list) else [parsed]
        except (json.JSONDecodeError, TypeError):
            turns = [text]

    return "\n".join(str(turn) for turn in turns)


def prepare_inputs(frame):
    out = frame.copy()

    out["prompt_text"] = out["prompt"].map(parse_turns)
    out["response_a_text"] = out["response_a"].map(parse_turns)
    out["response_b_text"] = out["response_b"].map(parse_turns)

    return out


train_proc = prepare_inputs(train_aug)
valid_proc = prepare_inputs(valid_part)
test_proc = prepare_inputs(test_df)


## 6. Tokenization and balanced truncation

Maximum input length is **1024 tokens**. The prompt is capped at 256 tokens; the remaining budget is shared between responses A and B. If one response is short, its unused budget is given to the other response.

When a field must be truncated, tokens are kept from both its beginning and end instead of dropping the tail completely.


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if USE_FP16 else None,
)

assert tokenizer.cls_token_id is not None
assert tokenizer.sep_token_id is not None

PROMPT_PREFIX_IDS = tokenizer(
    "PROMPT:\n",
    add_special_tokens=False,
)["input_ids"]

RESPONSE_A_PREFIX_IDS = tokenizer(
    "\n\nRESPONSE A:\n",
    add_special_tokens=False,
)["input_ids"]

RESPONSE_B_PREFIX_IDS = tokenizer(
    "\n\nRESPONSE B:\n",
    add_special_tokens=False,
)["input_ids"]


def head_tail_truncate(token_ids, budget, tail_fraction=TAIL_FRACTION):
    """Keep the beginning and end of a token sequence."""
    if budget <= 0:
        return []

    if len(token_ids) <= budget:
        return token_ids

    if budget == 1:
        return token_ids[:1]

    tail_size = max(1, int(round(budget * tail_fraction)))
    tail_size = min(tail_size, budget - 1)
    head_size = budget - tail_size

    return token_ids[:head_size] + token_ids[-tail_size:]


def balanced_response_budgets(len_a, len_b, total_budget):
    """Split the available response budget without starving either response."""
    if len_a + len_b <= total_budget:
        return len_a, len_b

    half = total_budget // 2

    if len_a <= half:
        a_budget = len_a
        b_budget = min(len_b, total_budget - a_budget)
        return a_budget, b_budget

    if len_b <= half:
        b_budget = len_b
        a_budget = min(len_a, total_budget - b_budget)
        return a_budget, b_budget

    # Both responses are long: split almost equally.
    # Give the possible extra token to the longer response.
    if len_a >= len_b:
        a_budget = total_budget - half
        b_budget = half
    else:
        a_budget = half
        b_budget = total_budget - half

    return a_budget, b_budget


SPECIAL_TOKEN_OVERHEAD = 1 + 3  # [CLS] + three [SEP] tokens
PREFIX_OVERHEAD = (
    len(PROMPT_PREFIX_IDS)
    + len(RESPONSE_A_PREFIX_IDS)
    + len(RESPONSE_B_PREFIX_IDS)
)
CONTENT_BUDGET = MAX_LENGTH - SPECIAL_TOKEN_OVERHEAD - PREFIX_OVERHEAD

assert CONTENT_BUDGET > MAX_PROMPT_TOKENS


def build_smart_input(prompt_ids, response_a_ids, response_b_ids):
    prompt_budget = min(
        len(prompt_ids),
        MAX_PROMPT_TOKENS,
        CONTENT_BUDGET,
    )

    response_budget = CONTENT_BUDGET - prompt_budget

    a_budget, b_budget = balanced_response_budgets(
        len(response_a_ids),
        len(response_b_ids),
        response_budget,
    )

    prompt_ids = head_tail_truncate(prompt_ids, prompt_budget)
    response_a_ids = head_tail_truncate(response_a_ids, a_budget)
    response_b_ids = head_tail_truncate(response_b_ids, b_budget)

    input_ids = (
        [tokenizer.cls_token_id]
        + PROMPT_PREFIX_IDS
        + prompt_ids
        + [tokenizer.sep_token_id]
        + RESPONSE_A_PREFIX_IDS
        + response_a_ids
        + [tokenizer.sep_token_id]
        + RESPONSE_B_PREFIX_IDS
        + response_b_ids
        + [tokenizer.sep_token_id]
    )

    assert len(input_ids) <= MAX_LENGTH

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
    }


def tokenize_batch(batch):
    prompt_batch = tokenizer(
        batch["prompt_text"],
        add_special_tokens=False,
        truncation=False,
        padding=False,
    )["input_ids"]

    response_a_batch = tokenizer(
        batch["response_a_text"],
        add_special_tokens=False,
        truncation=False,
        padding=False,
    )["input_ids"]

    response_b_batch = tokenizer(
        batch["response_b_text"],
        add_special_tokens=False,
        truncation=False,
        padding=False,
    )["input_ids"]

    encoded = [
        build_smart_input(prompt_ids, response_a_ids, response_b_ids)
        for prompt_ids, response_a_ids, response_b_ids
        in zip(prompt_batch, response_a_batch, response_b_batch)
    ]

    return {
        "input_ids": [item["input_ids"] for item in encoded],
        "attention_mask": [item["attention_mask"] for item in encoded],
    }


INPUT_COLUMNS = [
    "prompt_text",
    "response_a_text",
    "response_b_text",
]


def make_labeled_dataset(frame):
    ds = Dataset.from_pandas(
        frame[INPUT_COLUMNS + ["labels"]],
        preserve_index=False,
    )
    return ds.map(
        tokenize_batch,
        batched=True,
        remove_columns=INPUT_COLUMNS,
    )


def make_unlabeled_dataset(frame):
    ds = Dataset.from_pandas(
        frame[INPUT_COLUMNS],
        preserve_index=False,
    )
    return ds.map(
        tokenize_batch,
        batched=True,
        remove_columns=INPUT_COLUMNS,
    )


train_dataset = make_labeled_dataset(train_proc)
valid_dataset = make_labeled_dataset(valid_proc)

print(train_dataset)
print(valid_dataset)

train_lengths = np.asarray([len(ids) for ids in train_dataset["input_ids"]])
valid_lengths = np.asarray([len(ids) for ids in valid_dataset["input_ids"]])

print(
    "Train token lengths:",
    {
        "median": float(np.median(train_lengths)),
        "p90": float(np.percentile(train_lengths, 90)),
        "p95": float(np.percentile(train_lengths, 95)),
        "max": int(train_lengths.max()),
        "at_1024": float(np.mean(train_lengths == MAX_LENGTH)),
    },
)
print(
    "Valid token lengths:",
    {
        "median": float(np.median(valid_lengths)),
        "p90": float(np.percentile(valid_lengths, 90)),
        "p95": float(np.percentile(valid_lengths, 95)),
        "max": int(valid_lengths.max()),
        "at_1024": float(np.mean(valid_lengths == MAX_LENGTH)),
    },
)


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/103458 [00:00<?, ? examples/s]

Map:   0%|          | 0/5748 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 103458
})
Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 5748
})
Train token lengths: {'median': 524.0, 'p90': 1024.0, 'p95': 1024.0, 'max': 1024, 'at_1024': 0.1470355119952058}
Valid token lengths: {'median': 520.0, 'p90': 1024.0, 'p95': 1024.0, 'max': 1024, 'at_1024': 0.14979123173277661}


## 7. Model and metrics

DeBERTa-v3-base is fine-tuned with a 3-class classification head. The runtime position buffer is extended to 1024 tokens, while the pretrained relative-position range remains 512 tokens.

This avoids creating a new 1024-token relative-position range from scratch; the checkpoint still uses its learned relative-position buckets beyond the original window.


In [8]:
def softmax_numpy(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax_numpy(logits)
    preds = probs.argmax(axis=1)

    return {
        "log_loss": log_loss(labels, probs, labels=[0, 1, 2]),
        "accuracy": accuracy_score(labels, preds),
    }


def build_model(classifier_dropout):
    config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=3)

    # DeBERTa-v3-base has no absolute position embeddings
    # (position_biased_input=False), so extending this value only enlarges
    # the runtime position-id buffer. Keep the relative-position range
    # at the pretrained 512-token setting.
    config.max_position_embeddings = MAX_LENGTH
    config.max_relative_positions = PRETRAINED_RELATIVE_WINDOW

    config.id2label = {
        0: "winner_model_a",
        1: "winner_model_b",
        2: "winner_tie",
    }
    config.label2id = {
        "winner_model_a": 0,
        "winner_model_b": 1,
        "winner_tie": 2,
    }
    config.classifier_dropout = float(classifier_dropout)
    config.cls_dropout = float(classifier_dropout)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        config=config,
        dtype=torch.float32,
    )

    return model.float()

## 8. Hyperparameter tuning

Hyperparameter tuning is disabled for this run. Set `RUN_HP_TUNING = True` in Setup to run the search again.


In [9]:
def subset_dataset(dataset, fraction, seed):
    if fraction >= 1.0:
        return dataset

    size = max(1, int(len(dataset) * fraction))
    return dataset.shuffle(seed=seed).select(range(size))


best_hp = {
    "learning_rate": 1e-5,
    "classifier_dropout": 0.1,
    "label_smoothing_factor": 0.05,
}

if RUN_HP_TUNING:
    hp_train = subset_dataset(train_dataset, HP_TRAIN_FRACTION, SEED)
    hp_valid = subset_dataset(valid_dataset, HP_VALID_FRACTION, SEED + 1)

    results = []

    print("HP train rows:", len(hp_train))
    print("HP valid rows:", len(hp_valid))

    for trial_idx, hp in enumerate(HP_SEARCH_SPACE, start=1):
        print(f"\nTrial {trial_idx}/{len(HP_SEARCH_SPACE)}: {hp}")

        set_seed(SEED + trial_idx)
        trial_model = build_model(hp["classifier_dropout"])

        trial_args = TrainingArguments(
            output_dir=str(OUTPUT_DIR / f"hp_trial_{trial_idx}"),
            eval_strategy="epoch",
            save_strategy="no",
            num_train_epochs=HP_EPOCHS,
            per_device_train_batch_size=TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=EVAL_BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM_STEPS,
            learning_rate=hp["learning_rate"],
            weight_decay=WEIGHT_DECAY,
            lr_scheduler_type="cosine",
            label_smoothing_factor=hp["label_smoothing_factor"],
            fp16=USE_FP16,
            gradient_checkpointing=True,
            group_by_length=True,
            logging_steps=100,
            report_to="none",
            seed=SEED + trial_idx,
            data_seed=SEED + trial_idx,
        )

        trial_trainer = Trainer(
            model=trial_model,
            args=trial_args,
            train_dataset=hp_train,
            eval_dataset=hp_valid,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
        )

        trial_trainer.train()
        metrics = trial_trainer.evaluate()

        results.append({
            **hp,
            "eval_log_loss": float(metrics["eval_log_loss"]),
            "eval_accuracy": float(metrics["eval_accuracy"]),
        })

        del trial_trainer, trial_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    hp_results = pd.DataFrame(results).sort_values("eval_log_loss").reset_index(drop=True)
    hp_results.to_csv(HP_RESULTS_PATH, index=False)

    winner = hp_results.iloc[0]
    best_hp = {
        "learning_rate": float(winner["learning_rate"]),
        "classifier_dropout": float(winner["classifier_dropout"]),
        "label_smoothing_factor": float(winner["label_smoothing_factor"]),
    }

    display(hp_results)

with open(BEST_HP_PATH, "w", encoding="utf-8") as f:
    json.dump(best_hp, f, indent=2)

print("Best hyperparameters:", best_hp)

Best hyperparameters: {'learning_rate': 1e-05, 'classifier_dropout': 0.1, 'label_smoothing_factor': 0.05}


## 9. Final fine-tuning

The selected configuration is trained on the full swap-augmented training set.

Checkpoints are saved during training, and the final model and tokenizer are saved to `/kaggle/working/`.

In [10]:
from pathlib import Path

ckpt = Path(RESUME_FROM_CHECKPOINT)

assert ckpt.exists()
assert (ckpt / "model.safetensors").exists()
assert (ckpt / "optimizer.pt").exists()
assert (ckpt / "scheduler.pt").exists()
assert (ckpt / "trainer_state.json").exists()

print("Resume checkpoint:", ckpt)

Resume checkpoint: /kaggle/working/checkpoint-3000-fixed


In [11]:
print("Using hyperparameters:", best_hp)

set_seed(SEED)
model = build_model(best_hp["classifier_dropout"])

training_args = TrainingArguments(
    warmup_steps=100,
    output_dir=str(CHECKPOINT_DIR),
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="log_loss",
    greater_is_better=False,

    num_train_epochs=FINAL_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=best_hp["learning_rate"],
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    label_smoothing_factor=best_hp["label_smoothing_factor"],

    fp16=USE_FP16,
    gradient_checkpointing=True,
    group_by_length=True,
    logging_steps=500,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
trainer.save_state()

print("Saved model:", FINAL_MODEL_DIR)

Using hyperparameters: {'learning_rate': 1e-05, 'classifier_dropout': 0.1, 'label_smoothing_factor': 0.05}


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight        

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model: /kaggle/working/deberta-v3-base-cross-encoder-1024-smart


## 10. Validation

Evaluate the fine-tuned model on the untouched validation split.

In [12]:
validation_metrics = trainer.evaluate()

print({
    "log_loss": validation_metrics["eval_log_loss"],
    "accuracy": validation_metrics["eval_accuracy"],
})

{'log_loss': 1.0186488861440572, 'accuracy': 0.48521224773834376}


## 11. Test inference

Generate class probabilities for `test.csv` and save `submission.csv`.

In [13]:
test_dataset = make_unlabeled_dataset(test_proc)

test_logits = trainer.predict(test_dataset).predictions
test_probs = softmax_numpy(test_logits)

submission = pd.DataFrame({
    "id": test_df["id"].to_numpy(),
    "winner_model_a": test_probs[:, 0],
    "winner_model_b": test_probs[:, 1],
    "winner_tie": test_probs[:, 2],
})

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
display(submission.head())

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Saved: /kaggle/working/submission.csv


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.294589,0.384794,0.320617
1,211333,0.219856,0.489728,0.290416
2,1233961,0.278800,0.275328,0.445873
